In [9]:
import requests
from bs4 import BeautifulSoup
from datetime import datetime
import pandas as pd
from pathlib import Path


# Config
WEEKLY_URL = "https://kworb.net/spotify/country/global_weekly.html"
CSV_WEEKLY = Path("spotify_weekly_kworb.csv")
CSV_MONTHLY = Path("spotify_monthly_kworb.csv")
TOP_N = 50

def now_iso():
    return datetime.now().strftime("%Y-%m-%dT%H:%M:%S")

# Scrape Kworb weekly chart
def scrape_kworb_weekly(top_n=TOP_N, url=WEEKLY_URL):
    headers = {"User-Agent": "Mozilla/5.0"}
    resp = requests.get(url, headers=headers)
    resp.raise_for_status()

    soup = BeautifulSoup(resp.text, "html.parser")
    table = soup.find("table")
    if not table:
        print("No table found on Kworb page!")
        return pd.DataFrame()

    rows = table.find_all("tr")[1:top_n+1]
    data = []
    timestamp = now_iso()

    for row in rows:
        cols = [c.get_text(strip=True) for c in row.find_all("td")]
        if len(cols) >= 5:
            rank = int(cols[0])
            track = cols[1]
            artist = cols[2]
            streams = int(cols[3].replace(",", ""))
            data.append({
                "timestamp": timestamp,
                "rank": rank,
                "track": track,
                "artist": artist,
                "streams": streams
            })

    return pd.DataFrame(data)

# Save weekly snapshot
def save_weekly(df):
    if df.empty:
        print("No data to save.")
        return

    if CSV_WEEKLY.exists():
        df.to_csv(CSV_WEEKLY, mode='a', header=False, index=False)
    else:
        df.to_csv(CSV_WEEKLY, index=False)
    print(f"Weekly snapshot saved ({len(df)} rows).")

# Aggregate monthly from weekly
def aggregate_monthly():
    if not CSV_WEEKLY.exists():
        print("No weekly CSV found!")
        return pd.DataFrame()

    df = pd.read_csv(CSV_WEEKLY, parse_dates=["timestamp"])
    df["month"] = df["timestamp"].dt.to_period("M")  # YYYY-MM

    # Aggregate by month, track, artist
    monthly = df.groupby(["month", "track", "artist"]).agg({
        "streams": "sum",  # sum streams across weeks
        "rank": "min"      # best rank achieved in month
    }).reset_index()

    # Determine new releases
    monthly["is_new_release"] = False
    all_weekly = df.sort_values("timestamp")

    # find first month seen for each track
    first_seen = all_weekly.groupby(["track", "artist"]).timestamp.min().dt.to_period("M").reset_index()
    first_seen.rename(columns={"timestamp": "first_month"}, inplace=True)

    # mark as new release if current month == first month seen
    monthly = monthly.merge(first_seen, on=["track", "artist"], how="left")
    monthly["is_new_release"] = monthly["month"] == monthly["first_month"]

    # Cleanup
    monthly.drop(columns=["first_month"], inplace=True)

    monthly.to_csv(CSV_MONTHLY, index=False)
    print(f"Monthly aggregation saved ({len(monthly)} rows).")
    return monthly

# Run scraper + aggregation
weekly_df = scrape_kworb_weekly()
save_weekly(weekly_df)

monthly_df = aggregate_monthly()
print("\nTop rows of monthly track aggregation:")
print(monthly_df.head())


Weekly snapshot saved (50 rows).
Monthly aggregation saved (50 rows).

Top rows of monthly track aggregation:
     month track                                             artist  streams  \
0  2026-02    +1                               Alex Warren-Ordinary      208   
1  2026-02    +1                                    Coldplay-Yellow     1072   
2  2026-02    +1  El Bogueto-Cuando No Era Cantante - Remix(w/An...       32   
3  2026-02    +1                   Taylor Swift-The Fate of Ophelia       72   
4  2026-02    +1  The Weeknd-One Of The Girls(w/JENNIE,Lily-Rose...      488   

   rank  is_new_release  
0    12            True  
1    45            True  
2    36            True  
3     2            True  
4    38            True  
